# Object Detection
`capabilities/detection/grounding_dino.py`  
`capabilities/detection/owlvit_detector.py`  
`capabilities/detection/yolo_detector.py`

Three detectors, each suited to different scenarios:

| Class | Vocab | Speed (CPU) | Size | When to use |
|-------|-------|-------------|------|-------------|
| `GroundingDINODetector` | Open (text) | 5–8s | 700 MB | Best accuracy, GPU recommended |
| `OWLViTDetector` | Open (text) | 2–3s | 200 MB | CPU-friendly open-vocab |
| `YOLODetector` | COCO (80 classes) | < 50ms | 6 MB | Ultra-fast, known classes |

All three return the same `DetectionResult` type — downstream code is detector-agnostic.

## Setup

In [ ]:
import sys, os, tempfile
sys.path.insert(0, os.path.abspath('../..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from capabilities.detection.base_detection_result import Detection, DetectionResult
from sprout_detection.utils.image_gen import make_sprout_image

tmp = tempfile.mkdtemp()
plant_path = os.path.join(tmp, 'plant.png')
cv2.imwrite(plant_path, make_sprout_image(seed=42))

print('Test image created')

## DetectionResult — the shared result type

In [ ]:
# Explore the DetectionResult and Detection types without loading any model
img = cv2.imread(plant_path)
h, w = img.shape[:2]

# Build a mock result to show the API
mock_detection = Detection(
    label='green sprout',
    confidence=0.78,
    bbox=(w*0.2, h*0.1, w*0.8, h*0.9),
    bbox_normalised=(0.2, 0.1, 0.8, 0.9),
    area_px=(w*0.6) * (h*0.8),
    area_ratio=0.48,
)

mock_result = DetectionResult(
    image_path=plant_path,
    model_name='demo',
    detections=[mock_detection],
    prompts=['green sprout'],
    image_width=w,
    image_height=h,
)

print('DetectionResult API:')
print(f'  result.count           : {mock_result.count}')
print(f'  result.found           : {mock_result.found}')
print(f'  result.best.label      : {mock_result.best.label}')
print(f'  result.best.confidence : {mock_result.best.confidence}')
print(f'  result.best.bbox       : {tuple(round(v,1) for v in mock_result.best.bbox)}')
print(f'  result.best.centre     : {tuple(round(v,1) for v in mock_result.best.centre)}')
print(f'  result.best.width      : {mock_result.best.width:.1f}px')
print(f'  result.best.height     : {mock_result.best.height:.1f}px')
print(f'  result.best.area_ratio : {mock_result.best.area_ratio:.2f}')
print()
print('Filtering methods:')
print(f'  result.filter_by_label("green sprout")      : {len(mock_result.filter_by_label("green sprout"))} detections')
print(f'  result.filter_by_confidence(0.5)            : {len(mock_result.filter_by_confidence(0.5))} detections')

## Visualise bounding boxes

In [ ]:
def draw_detections(image_path, result, title='Detections'):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    ax.imshow(img)
    
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    for i, det in enumerate(result.detections):
        x1, y1, x2, y2 = det.bbox
        col = colors[i % len(colors)]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=col, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{det.label} {det.confidence:.2f}",
                color='white', fontsize=8, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=col, alpha=0.8))
    
    ax.set_title(f'{title} — {result.count} detection(s)', fontsize=10)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

draw_detections(plant_path, mock_result, title='Mock Detection (Demo)')

---
## OWLViTDetector — open-vocabulary, CPU-friendly

> Downloads ~200 MB on first use. Requires `transformers` and `torch`.

In [ ]:
from capabilities.detection.owlvit_detector import OWLViTDetector

detector = OWLViTDetector(score_threshold=0.08)
print(repr(detector))
print('Model loads on first detect() call')

In [ ]:
# OWL-ViT works best with noun phrases prefixed by "a"
result = detector.detect(
    plant_path,
    prompts=['a green seedling', 'a plant sprout', 'bare soil']
)

print(f'Detections: {result.count}')
print(f'Found:      {result.found}')
print(f'Time:       {result.duration_ms:.0f} ms')
print()
for d in result.detections:
    print(f'  {d.label:<20} conf={d.confidence:.3f}  area={d.area_ratio:.3f}')

if result.found:
    draw_detections(plant_path, result, title='OWL-ViT Detection')

## GroundingDINODetector — highest accuracy

> Downloads ~700 MB on first use. GPU recommended.

In [ ]:
from capabilities.detection.grounding_dino import GroundingDINODetector

dino = GroundingDINODetector(score_threshold=0.3)
print(repr(dino))

In [ ]:
# DINO accepts natural text descriptions separated by " . "
result_dino = dino.detect(
    plant_path,
    prompts=['green plant sprout', 'seedling']
)

print(f'Detections: {result_dino.count}  |  Time: {result_dino.duration_ms:.0f} ms')
for d in result_dino.detections:
    print(f'  {d.label:<20} conf={d.confidence:.3f}  bbox={tuple(round(v) for v in d.bbox)}')

if result_dino.found:
    draw_detections(plant_path, result_dino, title='Grounding DINO Detection')

## YOLODetector — ultra-fast, COCO classes

> Downloads ~6 MB (nano) on first use. Requires `ultralytics`.

In [ ]:
from capabilities.detection.yolo_detector import YOLODetector

yolo = YOLODetector(model_variant='yolov8n.pt', confidence_threshold=0.3)
print(repr(yolo))

In [ ]:
# YOLO detects 80 COCO classes
result_yolo = yolo.detect(plant_path)

print(f'Detections: {result_yolo.count}  |  Time: {result_yolo.duration_ms:.0f} ms')
for d in result_yolo.detections:
    print(f'  {d.label:<20} conf={d.confidence:.3f}')

# Filter to specific classes
result_filtered = yolo.detect(plant_path, filter_classes=['potted plant', 'vase'])
print(f'\nFiltered to pots/vases: {result_filtered.count} detections')

print(f'\nAll 80 COCO classes:')
print(yolo.coco_classes)

## All detectors return the same result type — swap freely

In [ ]:
# This function works with any detector that returns DetectionResult
def count_plants(image_path, detector, prompts):
    """Count plants in an image using any detector."""
    result = detector.detect(image_path, prompts=prompts)
    return result.count, result.best.confidence if result.found else 0.0

# Use with OWL-ViT (cpu friendly)
count, conf = count_plants(plant_path, detector, prompts=['a green sprout'])
print(f'OWL-ViT  → {count} plant(s), best conf = {conf:.3f}')

# Swap to DINO with zero code change
count, conf = count_plants(plant_path, dino, prompts=['green sprout'])
print(f'DINO     → {count} plant(s), best conf = {conf:.3f}')

## JSON serialisation

In [ ]:
import json
print(mock_result.to_json())